# Data Processing — OpenFace2 Results to Analysis-Ready CSVs

In [ ]:
import os
import pandas as pd
from pathlib import Path
from collections import defaultdict

# Input: synchronized 50-fps CSVs produced by fix_videos_openface2.ipynb
BASE_PATH = "..."

# Output: column-filtered + time-cut CSVs
OUTPUT_CUT = Path("...")

FPS = 50

input_paths = sorted(
    os.path.join(BASE_PATH, name)
    for name in os.listdir(BASE_PATH)
    if name.startswith("diti") and os.path.isdir(os.path.join(BASE_PATH, name))
)

print(f"Found {len(input_paths)} session folders in {BASE_PATH}")


## Step 1 — Parse session folders and group by meeting

In [ ]:
sessions = []
missing_info = []

for session_dir in input_paths:
    basename = os.path.basename(session_dir)
    parts = basename.split("_")
    if len(parts) < 3 or not parts[2].isdigit():
        missing_info.append((session_dir, "Unable to parse session index"))
        continue
    meeting_id = "_".join(parts[:2])
    session_idx = int(parts[2])
    infant_csv = os.path.join(session_dir, "Camera 1", "infant.csv")
    adult_csv = os.path.join(session_dir, "Camera 2", "adult.csv")
    if not os.path.exists(infant_csv) or not os.path.exists(adult_csv):
        missing_info.append((session_dir, "Missing infant/adult CSV"))
        continue
    sessions.append((meeting_id, session_idx, infant_csv, adult_csv))

if missing_info:
    print("Missing data entries detected:")
    for path, reason in missing_info:
        print(f"  - {path}: {reason}")

meeting_groups = defaultdict(list)
for meeting_id, session_idx, infant_csv, adult_csv in sessions:
    meeting_groups[meeting_id].append((session_idx, infant_csv, adult_csv))

for meeting_id in meeting_groups:
    meeting_groups[meeting_id].sort(key=lambda item: item[0])

# Only keep meetings with all 3 periods
removed_meetings = [mid for mid, entries in meeting_groups.items() if len(entries) != 3]
if removed_meetings:
    print("Excluding meetings without all three sessions:")
    for mid in sorted(removed_meetings):
        available_sessions = [session_idx for session_idx, _, _ in meeting_groups[mid]]
        print(f"  - {mid}: sessions present {available_sessions}")

meeting_groups = {mid: entries for mid, entries in meeting_groups.items() if len(entries) == 3}

print(f"\nPrepared groups for {len(meeting_groups)} meetings (only complete sets).")

## Step 2 — Load CSVs and column-filter

In [ ]:
# Column filtering
base_keep_cols = {"frame", "Frame", "face_id", "timestamp", "confidence", "success"}

def select_relevant_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only metadata + AU columns, dropping everything else."""
    keep_columns = [col for col in df.columns if col in base_keep_cols or "AU" in col]
    return df.loc[:, keep_columns]

# Load all meetings

df_infant_arr = {}
df_adult_arr = {}

for meeting_id, entries in meeting_groups.items():
    infant_dfs = []
    adult_dfs = []
    for _, infant_csv, adult_csv in entries:
        infant_dfs.append(select_relevant_columns(pd.read_csv(infant_csv)))
        adult_dfs.append(select_relevant_columns(pd.read_csv(adult_csv)))
    df_infant_arr[meeting_id] = infant_dfs
    df_adult_arr[meeting_id] = adult_dfs

# Sanity check
for meeting_id in df_infant_arr:
    for idx, (idf, adf) in enumerate(
        zip(df_infant_arr[meeting_id], df_adult_arr[meeting_id]), start=1
    ):
        if len(idf) != len(adf):
            print(
                f"WARNING row-count mismatch: {meeting_id} period {idx}: "
                f"infant={len(idf)}, adult={len(adf)}"
            )

print(f"Loaded data for {len(df_infant_arr)} meetings (column-filtered).")

## Step 3 — Apply time cuts and write

In [ ]:
cut_ranges = {
    ...
}


def time_str_to_seconds(time_str):
    """Convert 'MM:SS' string to total seconds."""
    minutes, seconds = time_str.split(":")
    return int(minutes) * 60 + int(seconds)


def cut_pair_by_time(infant_df, adult_df, start_time_str, end_time_str):
    """Cut both DataFrames to a time range, preserving row alignment."""
    start_sec = time_str_to_seconds(start_time_str)
    end_sec = time_str_to_seconds(end_time_str)

    # Determine row indices from the infant's timestamps
    mask = (infant_df['timestamp'] >= start_sec) & (infant_df['timestamp'] <= end_sec)
    indices = infant_df.index[mask]

    if len(indices) == 0:
        raise ValueError(
            f"No infant rows found in [{start_time_str}, {end_time_str}]"
        )

    # Slice both DataFrames by the SAME row indices
    cut_infant = infant_df.loc[indices].copy().reset_index(drop=True)
    cut_adult = adult_df.loc[indices].copy().reset_index(drop=True)

    # Re-index frame numbers and timestamps
    n = len(cut_infant)
    dt = 1.0 / FPS  # 0.02 for 50 fps

    for df in (cut_infant, cut_adult):
        if 'frame' in df.columns:
            df['frame'] = range(1, n + 1)
        if 'Frame' in df.columns:
            df['Frame'] = range(n)
        df['timestamp'] = [round(i * dt, 4) for i in range(n)]

    return cut_infant, cut_adult


# Write cut output

OUTPUT_CUT.mkdir(parents=True, exist_ok=True)

cut_write_count = 0
sessions_cut = []

for meeting_id, entries in meeting_groups.items():
    for idx, (session_idx, infant_csv, adult_csv) in enumerate(entries):
        # Derive the actual folder name (handles both "01" and "1" formats)
        session_dir_name = os.path.basename(os.path.dirname(os.path.dirname(infant_csv)))

        infant_df = df_infant_arr[meeting_id][idx].copy()
        adult_df = df_adult_arr[meeting_id][idx].copy()

        # Apply time cut if this session is in cut_ranges
        if session_dir_name in cut_ranges:
            start_time, end_time = cut_ranges[session_dir_name]
            infant_df, adult_df = cut_pair_by_time(
                infant_df, adult_df, start_time, end_time
            )
            sessions_cut.append(session_dir_name)
            print(f"Cut {session_dir_name}: {start_time} -> {end_time} "
                  f"({len(infant_df)} rows)")

        # Assert alignment before writing
        assert len(infant_df) == len(adult_df), (
            f"Row mismatch after cut: {session_dir_name} "
            f"infant={len(infant_df)}, adult={len(adult_df)}"
        )

        # Write to cut output directory (same folder structure as input)
        infant_rel = Path(infant_csv).relative_to(BASE_PATH)
        adult_rel = Path(adult_csv).relative_to(BASE_PATH)
        infant_target = OUTPUT_CUT / infant_rel
        adult_target = OUTPUT_CUT / adult_rel
        infant_target.parent.mkdir(parents=True, exist_ok=True)
        adult_target.parent.mkdir(parents=True, exist_ok=True)
        infant_df.to_csv(infant_target, index=False)
        adult_df.to_csv(adult_target, index=False)
        cut_write_count += 2

print(f"\nWrote {cut_write_count} CSV files to {OUTPUT_CUT}")
print(f"Sessions cut: {len(sessions_cut)}")
print(f"Cut sessions: {sorted(sessions_cut)}")